**1.1 Load the Required Libraries**

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from joblib import dump, load
from flask import Flask, jsonify

In [14]:
# Step 1: Load the Merged Dataset
# Replace 'merged_data.csv' with your actual file path
merged_data = pd.read_csv('merged_data.csv')

In [15]:
# Step 2: Handle Missing Values for Prices
# Replace missing prices with 0
merged_data[['Price_Aldi', 'Price_Coles', 'Price_IGA', 'Price_Woolworths']] = merged_data[
    ['Price_Aldi', 'Price_Coles', 'Price_IGA', 'Price_Woolworths']
].fillna(0)

In [16]:
# Step 3: Create the Target Column (`Discount_Flag`)
# Logic: Set Discount_Flag to 1 if any price is below a threshold (e.g., 5), otherwise 0
def define_discount_flag(row):
    threshold = 5  # Define the discount threshold
    if (row['Price_Aldi'] < threshold or
        row['Price_Coles'] < threshold or
        row['Price_IGA'] < threshold or
        row['Price_Woolworths'] < threshold):
        return 1
    return 0

merged_data['Discount_Flag'] = merged_data.apply(define_discount_flag, axis=1)

In [17]:
# Step 4: Prepare Features (`X`) and Target (`y`)
X = merged_data[['Price_Aldi', 'Price_Coles', 'Price_IGA', 'Price_Woolworths']].fillna(0)
y = merged_data['Discount_Flag']

In [18]:
# Verify the shapes of the features and target
print(f"Features Shape: {X.shape}")
print(f"Target Shape: {y.shape}")

Features Shape: (43870, 4)
Target Shape: (43870,)


In [19]:
# Step 5: Split the Data into Train and Test Sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [20]:
# Step 6: Train the Model
model = RandomForestClassifier(random_state=42, n_estimators=100)
model.fit(X_train, y_train)

RandomForestClassifier(random_state=42)

In [21]:
# Step 7: Evaluate the Model
y_test_pred = model.predict(X_test)
print(classification_report(y_test, y_test_pred))

              precision    recall  f1-score   support

           1       1.00      1.00      1.00      8774

    accuracy                           1.00      8774
   macro avg       1.00      1.00      1.00      8774
weighted avg       1.00      1.00      1.00      8774



In [22]:
# Step 8: Save the Model for Backend Integration
MODEL_FILE = 'discount_notification_model.joblib'
dump(model, MODEL_FILE)
print(f"Model saved to {MODEL_FILE}")

Model saved to discount_notification_model.joblib


**Model Testing**

In [23]:
# Import necessary libraries
import pandas as pd
import numpy as np
from joblib import load

In [24]:
# Step 1: Load the Trained Model
MODEL_FILE = 'discount_notification_model.joblib'
model = load(MODEL_FILE)
print(f"Model loaded from {MODEL_FILE}")

Model loaded from discount_notification_model.joblib


In [25]:
# Step 2: Prepare Test Data
# Replace this with actual test data or hypothetical test inputs
test_data = pd.DataFrame({
    'Price_Aldi': [3.5, 10.0, 2.0],  # Example test data
    'Price_Coles': [6.0, 5.0, 4.5],
    'Price_IGA': [8.0, 2.5, 6.0],
    'Price_Woolworths': [7.0, 9.0, 3.0]
})

In [26]:
print("Test Data:")
print(test_data)

Test Data:
   Price_Aldi  Price_Coles  Price_IGA  Price_Woolworths
0         3.5          6.0        8.0               7.0
1        10.0          5.0        2.5               9.0
2         2.0          4.5        6.0               3.0


In [27]:
# Step 3: Make Predictions
predictions = model.predict(test_data)
print("Predictions (Discount Flags):", predictions)

Predictions (Discount Flags): [1 1 1]


In [28]:
# Step 4: Interpret Results
# 1 means discount available; 0 means no discount
test_data['Discount_Flag'] = predictions
print("Test Data with Predictions:")
print(test_data)

Test Data with Predictions:
   Price_Aldi  Price_Coles  Price_IGA  Price_Woolworths  Discount_Flag
0         3.5          6.0        8.0               7.0              1
1        10.0          5.0        2.5               9.0              1
2         2.0          4.5        6.0               3.0              1
